# 03 — Zamana sıralı tahmin ve backtest

Model seçimi yalnızca 2023 doğrulama sonuçlarına dayanır. 2024 gözlemleri model ayarı veya değişken seçimi için kullanılmaz.

In [1]:
from pathlib import Path
import json, tomllib
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "config.toml").exists() else Path.cwd().parent
RAW, OUT = ROOT / "data/private/study-yahoo-real", ROOT / "results/research"
assert RAW.exists(), "Gerçek ham girdi data/private altında hazırlanmalıdır."
config = tomllib.loads((ROOT / "config.toml").read_text(encoding="utf-8"))
START, END, SECTORS = pd.Timestamp(config["study"]["start"]), pd.Timestamp(config["study"]["end"]), config["study"]["sectors"]
print(f"Çalışma dönemi: {START.date()} — {END.date()}")
print("Temsilciler:", ", ".join(SECTORS))

Çalışma dönemi: 2019-01-01 — 2024-12-31
Temsilciler: XBANK, XUSIN


In [2]:
from bist_risk.data import read_inputs, monthly_prices, align_macro, shock_scores
prices, macro, calendar, provenance = read_inputs(RAW, "research")
monthly = monthly_prices(prices, calendar, config["study"]["min_days"])
aligned = align_macro(macro, pd.date_range(monthly.date.min(), END, freq="ME"))
study_dates = pd.date_range(START, END, freq="ME")
quality_rows, eligible = [], []
for sector in SECTORS:
    block = monthly[(monthly.sector == sector) & monthly.date.between(START, END)].set_index("date").reindex(study_dates)
    complete = len(block) == len(study_dates) and block[["return_value", "volatility"]].notna().all().all()
    quality_rows.append({"sector": sector, "eligible": bool(complete), "months": int(block.return_value.count()), "reason": "complete" if complete else "missing daily sessions, warm-up or monthly data"})
    if complete: eligible.append(sector)
quality = pd.DataFrame(quality_rows)
assert eligible == SECTORS, quality

In [3]:
from bist_risk.modeling import supervised, training_rows, features_at, evaluate_sector, metrics
frames={s:supervised(monthly[(monthly.sector==s)&monthly.date.between(START,END)],aligned) for s in eligible}
example=frames[eligible[0]]; origin=pd.Timestamp("2023-01-31")
training=training_rows(example,origin,config["model"]["min_train"])
features=features_at(example,origin,"return",list(aligned.columns),config)
print("Örnek doğrulama kökeni:",origin.date())
print("Eğitim hedeflerinin son tarihi:",training.target_date.max().date())
print("Seçilen özellikler:",features)
assert training.target_date.max() <= origin

Örnek doğrulama kökeni: 2023-01-31
Eğitim hedeflerinin son tarihi: 2023-01-31
Seçilen özellikler: ['return_value', 'volatility']


## 2023 model seçimi ve 2024 son değerlendirmesi

Ridge ve XGBoost adayları 2023 genişleyen pencere doğrulamasıyla karşılaştırılır. Seçilen ayar, yalnızca 2023 sonuna kadar olan bilgiyle dondurulur.

In [4]:
prediction_rows,selection_rows=[],[]
for sector, frame in frames.items():
    sector_predictions,sector_selection=evaluate_sector(frame,list(aligned.columns),config,sector)
    prediction_rows.append(sector_predictions); selection_rows.append(sector_selection)
predictions=pd.concat(prediction_rows,ignore_index=True)
model_selection=pd.concat(selection_rows,ignore_index=True)
assert (predictions.fit_until <= predictions.origin).all()
assert (predictions.target_date.dt.year == config["study"]["test_year"]).all()
metrics_table=metrics(predictions,config["study"]["seed"])
display(model_selection); display(metrics_table)

,sector,target,model,setting,validation_mae
0,XBANK,return,ridge,1.0,0.100073
1,XBANK,return,ridge,10.0,0.099284
2,XBANK,return,xgboost,1.0,0.097431
3,XBANK,return,xgboost,2.0,0.108325
4,XBANK,volatility,ridge,1.0,0.139211
5,XBANK,volatility,ridge,10.0,0.135658
6,XBANK,volatility,xgboost,1.0,0.142627
7,XBANK,volatility,xgboost,2.0,0.145852
8,XUSIN,return,ridge,1.0,0.096407
9,XUSIN,return,ridge,10.0,0.095531


,sector,target,model,n,mae,rmse,mae_low,mae_high,baseline_mae,mae_improvement
0,XBANK,return,historical_mean,12,0.079159,0.100611,0.053450,0.106123,0.079159,0.000000
1,XBANK,return,persistence,12,0.112363,0.140772,0.065689,0.173808,0.079159,-0.419457
2,XBANK,return,ridge,12,0.076922,0.096858,0.050294,0.102955,0.079159,0.028268
3,XBANK,return,xgboost,12,0.072018,0.090658,0.046337,0.096769,0.079159,0.090209
4,XBANK,volatility,historical_mean,12,0.051355,0.061330,0.033805,0.074311,0.051355,0.000000
5,XBANK,volatility,persistence,12,0.076692,0.100177,0.045769,0.109533,0.051355,-0.493362
6,XBANK,volatility,ridge,12,0.067291,0.077527,0.047948,0.091090,0.051355,-0.310301
7,XBANK,volatility,xgboost,12,0.058866,0.068057,0.039154,0.084410,0.051355,-0.146255
8,XUSIN,return,historical_mean,12,0.059452,0.068857,0.043494,0.079491,0.059452,0.000000
9,XUSIN,return,persistence,12,0.073001,0.095483,0.047339,0.104297,0.059452,-0.227911


In [5]:
px.bar(metrics_table,x="sector",y="mae",color="model",facet_row="target",barmode="group",title="2024 son değerlendirme — MAE karşılaştırması").show()
fig=px.scatter(metrics_table,x="rmse",y="mae_improvement",color="model",symbol="target",hover_name="sector",title="RMSE ve geçmiş ortalamaya göre MAE iyileşmesi")
fig.add_hline(y=0,line_dash="dash",line_color="red"); fig.show()

In [6]:
from bist_risk.artifacts import write_tables
write_tables(OUT,{"predictions":predictions,"model_selection":model_selection,"metrics":metrics_table}); print("03 tahmin tabloları kaydedildi.")

03 tahmin tabloları kaydedildi.
